# SFT Flow


Use this notebook after the SFT flow note. The goal is to show the practical difference between a base checkpoint and a tiny supervised fine-tuning pass.

You will:
- evaluate a base model on assistant-formatted text
- fine-tune on one canonical chat schema
- compare the loss before and after SFT


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import copy
from course_tools import build_demo_bundle, evaluate_model, format_messages, train_model

LECTURE_NOTE_TITLE = 'SFT Flow'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: SFT Flow


## Start from a base checkpoint


In [2]:
bundle = build_demo_bundle(steps=20)
base_model = bundle['model']
tokenizer = bundle['tokenizer']
prompt_messages = [
    {'role': 'system', 'content': 'You are concise.'},
    {'role': 'user', 'content': 'What is self-attention?'},
]
target_chat = format_messages(prompt_messages + [
    {'role': 'assistant', 'content': 'Self-attention forms a weighted summary over relevant earlier tokens.'}
])
base_metrics = evaluate_model(base_model, tokenizer, text='\n'.join([target_chat] * 8), batch_size=4)
print('base eval on assistant-formatted target:', base_metrics)


base eval on assistant-formatted target: {'loss': 6.514034748077393, 'bpb': 9.397765627229642}


## Fine-tune on one canonical chat schema


In [3]:
sft_model = copy.deepcopy(base_model)
sft_corpus = '\n'.join([
    format_messages(prompt_messages + [{'role': 'assistant', 'content': 'Self-attention forms a weighted summary over relevant earlier tokens.'}]),
    format_messages([
        {'role': 'system', 'content': 'You are concise.'},
        {'role': 'user', 'content': 'What is tokenization?'},
        {'role': 'assistant', 'content': 'Tokenization maps text into discrete IDs.'},
    ]),
] * 6)
history = train_model(sft_model, tokenizer, train_text=sft_corpus, eval_text=sft_corpus, steps=12, learning_rate=2e-3, batch_size=4)
print(history)


[{'step': 1.0, 'train_loss': 6.070614337921143, 'eval_loss': 5.835428237915039, 'bpb': 8.418743380303448}, {'step': 2.0, 'train_loss': 6.302055835723877, 'eval_loss': 5.520715236663818, 'bpb': 7.964708494095031}, {'step': 4.0, 'train_loss': 5.212050437927246, 'eval_loss': 4.706051826477051, 'bpb': 6.78939763222489}, {'step': 6.0, 'train_loss': 4.834709644317627, 'eval_loss': 5.1449079513549805, 'bpb': 7.422533187250027}, {'step': 8.0, 'train_loss': 4.435783386230469, 'eval_loss': 4.485936164855957, 'bpb': 6.471837858782145}, {'step': 10.0, 'train_loss': 4.429383277893066, 'eval_loss': 4.158878803253174, 'bpb': 5.999993825111581}, {'step': 12.0, 'train_loss': 3.8465278148651123, 'eval_loss': 4.083313941955566, 'bpb': 5.89097677445206}]


## Compare the resulting checkpoint against the base model


In [4]:
sft_metrics = evaluate_model(sft_model, tokenizer, text='\n'.join([target_chat] * 8), batch_size=4)
print('sft eval on assistant-formatted target:', sft_metrics)
print('loss improvement:', round(base_metrics['loss'] - sft_metrics['loss'], 4))


sft eval on assistant-formatted target: {'loss': 3.8670566082000732, 'bpb': 5.578983391487141}
loss improvement: 2.647


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/gpt_instruction_finetuning.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/chat_sft.py
